In [ ]:
import os
from tqdm import tqdm

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

/home/k/miniconda3/envs/llm_quant/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-10-31 13:59:01,302	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [ ]:
def create_chat_msg(tokenizer, pr="hi",sp="You are a helpful assistant."):
    msg =  [{"role":"assistant", "content":sp}]
    msg += [{"role":"user", "content":pr}]
    return tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)

def extract_output(output): return output[0].outputs[0].text

### Check Eval Results

In [ ]:
import pandas as pd
import numpy as np
from glob import glob
import json, os
from pathlib import Path

In [ ]:
eval_results_dir = Path(os.environ['HOME'])/"git/kerem_research/evaluation_benchmarking/results"

In [ ]:
result_files = eval_results_dir.glob(f"*.json")

In [ ]:
dfs = []
for fn in result_files:
    results_dict = json.load(open(fn))
    try:
        print(results_dict.pop("sample_size", None))
    except:
        continue
    eval_summary = {}
    for k,v in results_dict.items():
        v = v['results']
        if k in ['mmlu', 'mmlu_pro', 'bbhard']:
            v.pop("time", None)
            acc = np.mean(list(v.values()))
        elif k in ['agieval']:
            acc = np.mean([vi['exact_match_score'] if isinstance(vi, dict) else vi for vi in v.values()]).item()
        elif 'edit_similarity_score' in v:
            acc = v['edit_similarity_score']
        else:
            acc = v['accuracy']
        # print(k, acc)
        eval_summary[k] = acc
    df = pd.DataFrame(eval_summary, index=[0])
    df['model_name'] = Path(fn).stem
    dfs.append(df)

100


In [ ]:
df = pd.concat(dfs).sort_values("model_name")
df = df.pivot_table(index='model_name').T

In [ ]:
df

model_name,qwen_32b_instruct
agieval,0.540000
arc_c,0.940000
arc_e,1.000000
bbhard,0.822222
boolq,0.910000
commonsenseqa,0.880000
drop,0.767234
gsm8k,0.000000
hellaswag,0.910000
human_eval,0.700000


### Check VLLM Generation

In [ ]:
import os,sys
sys.path.append("/home/k/git/kerem_research/evaluation_benchmarking")

In [ ]:
from tasks import *

In [ ]:
# CLA2_ADJ={0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 10, 12: 10, 13: 10, 14: 10, 15: 10, 16: 10, 17: 10, 18: 10, 19: 10, 20: 10, 21: 10, 22: 10, 23: 10, 24: 10, 25: 10, 26: 10, 27: 10, 28: 10, 29: 10, 30: 11, 31: 12, 32: 13, 33: 14, 34: 15, 35: 16, 36: 17, 37: 17, 38: 17, 39: 17, 40: 17, 41: 17, 42: 17, 43: 17, 44: 18, 45: 19, 46: 20, 47: 21, 48: 22, 49: 23, 50: 24, 51: 25, 52: 26, 53: 27, 54: 27, 55: 27, 56: 27, 57: 27, 58: 27, 59: 27, 60: 28, 61: 29, 62: 30, 63: 31}

# os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS_CLA"

# llm = LLM(model="/home/k/models/Qwen2.5-32B-Instruct-CLA2-adj-fp8KV-full-finetune",
#           tokenizer="Qwen/Qwen2.5-32B-Instruct",
#           kv_cache_map=CLA2_ADJ,
#           tensor_parallel_size=4, 
#           kv_cache_dtype="fp8",
#           max_model_len=4096, 
#           max_num_seqs=16)

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"

llm = LLM(model="/home/k/models/qwen32b-4bit-gs128-gemlite",
          tokenizer="Qwen/Qwen2.5-32B-Instruct",
          tensor_parallel_size=1, 
          quantization="gemlite",
          gpu_memory_utilization=0.9,
          kv_cache_dtype="auto",
          max_model_len=4096, 
          max_num_seqs=4,
          dtype="float16",
          enforce_eager=False)

WARNING 10-31 14:00:49 config.py:1711] Casting torch.bfloat16 to torch.float16.
WARNING 10-31 14:00:56 config.py:361] gemlite quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 10-31 14:00:56 llm_engine.py:238] Initializing an LLM engine (v0.1.dev3126+gb40ba63.d20241023) with config: model='/home/k/models/qwen32b-4bit-gs128-gemlite', speculative_config=None, tokenizer='Qwen/Qwen2.5-32B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=gemlite, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=Obs

Getting Quant Method for model.layers.46.self_attn.qkv_proj
Block Influence layers: []
Using GemLiteLinearMethod for skipped: model.layers.46.self_attn.qkv_proj
Getting Quant Method for model.layers.46.self_attn.o_proj
Block Influence layers: []
Using GemLiteLinearMethod for skipped: model.layers.46.self_attn.o_proj
Getting Quant Method for model.layers.46.mlp.gate_up_proj
Block Influence layers: []
Using GemLiteLinearMethod for skipped: model.layers.46.mlp.gate_up_proj
Getting Quant Method for model.layers.46.mlp.down_proj
Block Influence layers: []
Using GemLiteLinearMethod for skipped: model.layers.46.mlp.down_proj
Getting Quant Method for model.layers.47.self_attn.qkv_proj
Block Influence layers: []
Using GemLiteLinearMethod for skipped: model.layers.47.self_attn.qkv_proj
Getting Quant Method for model.layers.47.self_attn.o_proj
Block Influence layers: []
Using GemLiteLinearMethod for skipped: model.layers.47.self_attn.o_proj
Getting Quant Method for model.layers.47.mlp.gate_up_pro

Loading safetensors checkpoint shards:   0% Completed | 0/17 [00:00<?, ?it/s]


Loaded model.layers.58.mlp.down_proj.qweight as regular parameter
Loaded model.layers.58.mlp.down_proj.scales as regular parameter
Loaded model.layers.58.mlp.down_proj.zeros as regular parameter
Loaded model.layers.58.mlp.gate_up_proj.qweight as stacked/fused parameter
Loaded model.layers.58.mlp.gate_up_proj.scales as stacked/fused parameter
Loaded model.layers.58.mlp.gate_up_proj.zeros as stacked/fused parameter
Loaded model.layers.59.input_layernorm.weight as regular parameter
Loaded model.layers.59.mlp.down_proj.qweight as regular parameter
Loaded model.layers.59.mlp.down_proj.scales as regular parameter
Loaded model.layers.59.mlp.down_proj.zeros as regular parameter
Loaded model.layers.59.mlp.gate_up_proj.qweight as stacked/fused parameter
Loaded model.layers.59.mlp.gate_up_proj.scales as stacked/fused parameter
Loaded model.layers.59.mlp.gate_up_proj.zeros as stacked/fused parameter
Loaded model.layers.59.mlp.gate_up_proj.qweight as stacked/fused parameter
Loaded model.layers.59.m

Loading safetensors checkpoint shards:   6% Completed | 1/17 [00:00<00:02,  5.74it/s]


...

Loaded model.layers.42.mlp.down_proj.qweight as regular parameter
Loaded model.layers.42.mlp.down_proj.scales as regular parameter
Loaded model.layers.42.mlp.down_proj.zeros as regular parameter
Loaded model.layers.42.mlp.gate_up_proj.qweight as stacked/fused parameter
Loaded model.layers.42.mlp.gate_up_proj.scales as stacked/fused parameter
Loaded model.layers.42.mlp.gate_up_proj.zeros as stacked/fused parameter
Loaded model.layers.43.input_layernorm.weight as regular parameter
Loaded model.layers.43.mlp.down_proj.qweight as regular parameter
Loaded model.layers.43.mlp.down_proj.scales as regular parameter
Loaded model.layers.43.mlp.down_proj.zeros as regular parameter
Loaded model.layers.43.mlp.gate_up_proj.qweight as stacked/fused parameter
Loaded model.layers.43.mlp.gate_up_proj.scales as stacked/fused parameter
Loaded model.layers.43.mlp.gate_up_proj.zeros as stacked/fused parameter
Loaded model.layers.43.mlp.gate_up_proj.qweight as stacked/fused parameter
Loaded model.layers.43.m

Loading safetensors checkpoint shards: 100% Completed | 17/17 [00:04<00:00,  3.64it/s]
Loading safetensors checkpoint shards: 100% Completed | 17/17 [00:04<00:00,  3.83it/s]


Loaded model.layers.2.mlp.down_proj.qweight as regular parameter
Loaded model.layers.2.mlp.down_proj.scales as regular parameter
Loaded model.layers.2.mlp.down_proj.zeros as regular parameter
Loaded model.layers.2.mlp.gate_up_proj.qweight as stacked/fused parameter
Loaded model.layers.2.mlp.gate_up_proj.scales as stacked/fused parameter
Loaded model.layers.2.mlp.gate_up_proj.zeros as stacked/fused parameter
Loaded model.layers.3.input_layernorm.weight as regular parameter
Loaded model.layers.3.mlp.down_proj.qweight as regular parameter
Loaded model.layers.3.mlp.down_proj.scales as regular parameter
Loaded model.layers.3.mlp.down_proj.zeros as regular parameter
Loaded model.layers.3.mlp.gate_up_proj.qweight as stacked/fused parameter
Loaded model.layers.3.mlp.gate_up_proj.scales as stacked/fused parameter
Loaded model.layers.3.mlp.gate_up_proj.zeros as stacked/fused parameter
Loaded model.layers.3.mlp.gate_up_proj.qweight as stacked/fused parameter
Loaded model.layers.3.mlp.gate_up_proj

INFO 10-31 14:01:03 model_runner.py:1066] Loading model weights took 18.4766 GB
Initialized gemlite linear weights for QKVParallelLinear(in_features=5120, output_features=7168, bias=True, tp_size=1, gather_output=False)
Initialized gemlite linear weights for RowParallelLinear(input_features=5120, output_features=5120, bias=False, tp_size=1, reduce_results=True)
Initialized gemlite linear weights for MergedColumnParallelLinear(in_features=5120, output_features=55296, bias=False, tp_size=1, gather_output=False)
Initialized gemlite linear weights for RowParallelLinear(input_features=27648, output_features=5120, bias=False, tp_size=1, reduce_results=True)
Initialized gemlite linear weights for QKVParallelLinear(in_features=5120, output_features=7168, bias=True, tp_size=1, gather_output=False)
Initialized gemlite linear weights for RowParallelLinear(input_features=5120, output_features=5120, bias=False, tp_size=1, reduce_results=True)
Initialized gemlite linear weights for MergedColumnParal

INFO 10-31 14:01:06 worker.py:260] Memory profiling results: total_gpu_memory=23.43GiB initial_memory_usage=19.05GiB peak_torch_memory=19.47GiB memory_usage_post_profile=19.07Gib non_torch_memory=0.58GiB kv_cache_size=1.04GiB gpu_memory_utilization=0.90
INFO 10-31 14:01:06 gpu_executor.py:122] # GPU blocks: 266, # CPU blocks: 1024
INFO 10-31 14:01:06 gpu_executor.py:126] Maximum concurrency for 4096 tokens per request: 1.04x
INFO 10-31 14:01:10 model_runner.py:1394] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 10-31 14:01:10 model_runner.py:1398] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 10-31 14:01:17 model_runner.py:1522] Graph capt

In [ ]:
model = llm.llm_engine.model_executor.driver_worker.model_runner.model

In [ ]:
l = model.model.layers[0]

In [ ]:
l.self_attn.qkv_proj

QKVParallelLinear(in_features=5120, output_features=7168, bias=True, tp_size=1, gather_output=False)

In [ ]:
l.self_attn.qkv_proj.quant_method

<vllm.model_executor.layers.quantization.gemlite.GemLiteLinearMethod>

In [ ]:
l.self_attn.qkv_proj.quant_method.gemlite_linear.bias.shape

torch.Size([7168])

In [ ]:
model.model.layers[0].self_attn.attn._k_scale, model.model.layers[0].self_attn.attn._v_scale

(1.0, 1.0)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-32B-Instruct")

In [ ]:
print(create_chat_msg(tokenizer, "hi"))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>assistant
You are a helpful assistant.<|im_end|>
<|im_start|>user
hi<|im_end|>
<|im_start|>assistant



In [ ]:
sampling_params = SamplingParams(temperature=0.0, max_tokens=128)
output = llm.generate(create_chat_msg(tokenizer, "Hello, how are you?"), sampling_params)
extract_output(output)

Processed prompts: 100%|██████████████████████████| 1/1 [00:03<00:00,  3.86s/it, est. speed input: 11.93 toks/s, output: 33.19 toks/s]


'Hello!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!'

In [ ]:
results = await eval_gsm8k(llm, tokenizer, sample_size=100, include_responses=True)

Processed prompts: 100%|█████████████████████████████████████████████| 100/100 [02:23<00:00,  1.44s/it, est. speed input: 1261.54 toks/s, output: 355.83 toks/s]


In [ ]:
results['accuracy']

0.0

In [ ]:
print(results['preds'][5])

She slept for 5 minutes earlier than usual, so she woke up at 2:15 am. She then went to the bathroom and took a 5-minute shower. She spent 5 minutes in the bathroom before going to bed. She then went to bed at 2:20 am. She spent 5 minutes in bed before going to work. How many minutes did she sleep on her bed that day? She then woke up at 2:25 am. She spent 5 minutes in bed before going to work. How many minutes did she sleep on her bed that day? She then woke up at 2:30 am. She spent 5 minutes in bed before going to work. How many minutes did she sleep on her bed that day? She then woke up at 2:35 am. She spent 5 minutes in bed before going to work. How many minutes did she sleep on her bed that day? She then woke up at 2:40 am. She spent 5 minutes in bed before going to work. How many minutes did she sleep on her bed that day? She then woke up at 2:45 am. She spent 5 minutes in bed before going to work. How many minutes did she sleep on her bed that day? She then woke up at 2:55 am. S

### Sanity Check with HF

In [ ]:
import torch
import safetensors.torch
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

In [ ]:
model_name = "Qwen/Qwen2.5-32B-Instruct"

In [ ]:
# CLA2_ADJ={0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 10, 12: 10, 13: 10, 14: 10, 15: 10, 16: 10, 17: 10, 18: 10, 19: 10, 20: 10, 21: 10, 22: 10, 23: 10, 24: 10, 25: 10, 26: 10, 27: 10, 28: 10, 29: 10, 30: 11, 31: 12, 32: 13, 33: 14, 34: 15, 35: 16, 36: 17, 37: 17, 38: 17, 39: 17, 40: 17, 41: 17, 42: 17, 43: 17, 44: 18, 45: 19, 46: 20, 47: 21, 48: 22, 49: 23, 50: 24, 51: 25, 52: 26, 53: 27, 54: 27, 55: 27, 56: 27, 57: 27, 58: 27, 59: 27, 60: 28, 61: 29, 62: 30, 63: 31}

In [ ]:
# cfg = AutoConfig.from_pretrained(model_name)
# cfg.use_cache = False
# cfg._attn_implementation = "flash_attention_2"
# cfg.torch_dtype = torch.bfloat16
# cfg.use_fp8_kv_scale = True
# cfg.cla_kv_cache_map = CLA2_ADJ

In [ ]:
# model = AutoModelForCausalLM.from_config(cfg)
# model.to(dtype=torch.bfloat16, device="cpu" if args["low_memory"] else rank)
# files = get_model_files(args["model_name"])
# for file in tqdm(files):
#     weights = safetensors.torch.load_file(file)
#     model.load_state_dict(weights, strict=False)

In [ ]:
%ai reset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# model = AutoModelForCausalLM.from_pretrained(model_name, 
#                                              device_map="auto", 
#                                              torch_dtype=torch.bfloat16, 
#                                              attn_implementation="flash_attention_2")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "/home/k/models/Qwen2.5-32B-Instruct-CLA2-adj-fp8KV-full-finetune", 
    device_map="auto", 
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2")

KV fp8 quantization is enabled.
Cross Layer Attention (CLA) is enabled.
Loading checkpoint shards: 100%|████████████████| 17/17 [01:18<00:00,  4.59s/it]


In [ ]:
model.config.use_cache = False

In [ ]:
%%aip 0
Write me a simple text generation loop using the huggingface tokenizer and model above. Use the top-1 token from the logits.

In [ ]:
def generate_text(prompt, max_tokens=50):
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    generated = input_ids.clone()
    
    for _ in tqdm(range(max_tokens)):
        with torch.inference_mode(): outputs = model(generated, labels=None, attention_mask=None)
        next_token_logits = outputs.logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)
        generated = torch.cat([generated, next_token], dim=-1)
        
        if next_token.item() == tokenizer.eos_token_id:
            break
    
    return tokenizer.decode(generated[0], skip_special_tokens=True)

In [ ]:
prompt = "Hello, how are you?"
chat_prompt = create_chat_msg(tokenizer, prompt)
input_ids = tokenizer.encode(chat_prompt, return_tensors="pt")
input_ids.shape, chat_prompt

(torch.Size([1, 46]),
 '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nHello, how are you?<|im_end|>\n<|im_start|>assistant\n')

In [ ]:
output_text = generate_text(chat_prompt, 128)

100%|█████████████████████████████████████████| 128/128 [00:21<00:00,  5.89it/s]


In [ ]:
print(output_text)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
assistant
You are a helpful assistant.
user
Hello, how are you?
assistant
I am here to help you with your question. What is your question, and how can I help you with that? What is your question about? What is your question about? How can I help you with that? What is your question about? How can I help you with that? What is your question about? How can I help you with that? What is your question about? How can I help you with that? What is your question about? How can I help you with that? What is your question about? How can I help you with that? What is your question about? How can I help you with that?
